In [ ]:
from pathlib import Path
import yaml

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import shufflenet_v2_x0_5, ShuffleNet_V2_X0_5_Weights
from torchsummary import summary

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

# Moduli progetto
from models.model import ChimeraSeg
from models.decoder import Decoder
#from training.train import Trainer
from data.streethazards import StreetHazards
from utils.misc import get_device, fix_random

# Configurazioni di ambiente
%load_ext autoreload
%autoreload 2


Premessa: come il mio precedente notebook, link al notebook, sono interessato ad avere delle reti efficienti che mi permettano di poter trainare liberamente e fare diverse prove anche su un dispositivo mac con `mps`.

Guardando i diversi paper che ottenevano alti score nel benchmark SegmentMeIfYouCan, i migliori come mIoU utilizzano Mask classification e di conseguenza Mask2Former con l'aggiunta di un modulo o di tecniche per aggiungere anche il rilevamento di anomalie. Tra tutti ho preferito implementare il paper `Open Semantic Segmentation with Class Similarity` che ottiene un mIoU competitivo, rimanendo al contempo veloce a differenza degli approcci con Mask2Former più lenti di natura, e con un rilevamento di anomalie decisamente migliore. Successivamente, data la recente uscita del paper `Open Panoptic Segmentation`, sempre degli stessi autori, che oltre ad aggiungere la Panoptic Segmentation, risolve alcuni problemi del precedente approccio.

In [72]:
config_path = "./config.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

fix_random(config['seed'])
device = get_device()

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
transform = A.Compose([
    A.Resize(height=224, width=224),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [73]:
train_root = Path.home() / config['paths']['train_path']
print(train_root)

train_data = StreetHazards(
    train_root,
    "training",
    transforms=transform
)

val_data = StreetHazards(
    train_root,
    "validation"
)

classes = train_data.classes
num_classes = len(classes)

print(f"Number of training samples: {len(train_data)}")
print(f"Number of validation samples: {len(val_data)}")
print(f"Number of classes: {num_classes}")

/home/aarcara/ml4cv_assignment/data/datasets/train
Number of training samples: 5125
Number of validation samples: 1031
Number of classes: 14


In [55]:
train_loader = DataLoader(
    train_data,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_data,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    pin_memory=True
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")

Number of training batches: 21
Number of validation batches: 5


In [ ]:
from tqdm import tqdm

def calculate_class_weights(dataloader, num_classes):
    counts = torch.zeros(num_classes, dtype=torch.int64)

    for _, masks in tqdm(dataloader, desc="Calculating pixel counts"):
        counts += torch.bincount(masks.flatten(), minlength=num_classes)

    total_pixels = counts.sum().item()
    normalized_frequencies = counts.float() / total_pixels

    return normalized_frequencies.tolist()


calculate_class_weights(train_loader, num_classes)

Calculating pixel counts:  67%|██████▋   | 14/21 [01:11<00:37,  5.34s/it]

In [70]:
do_train = True 

pretrained_model = shufflenet_v2_x0_5(weights=ShuffleNet_V2_X0_5_Weights.DEFAULT)
backbone = nn.Sequential(*(list(pretrained_model.children())[:-1]))

decoder = Decoder(1024, 224)
model = ChimeraSeg(backbone, decoder)
trainer = Trainer(config, model, device, train_loader)

if do_train:
    trainer.train("debug")
else:
    pass

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [47]:
# from torchvision.utils import make_grid

# NUM_IMAGES = 6
# grid = make_grid([train_data[i][0] for i in range(NUM_IMAGES)], nrow=3)

# grid_image = grid.permute(1, 2, 0)

# plt.figure(figsize=(15, 5))
# plt.imshow(grid_image.numpy())
# plt.axis('off')
# plt.show()

In [48]:
# from utils.visualize import color, COLORS

# color(train_data[100][2], COLORS)

nel training devo vedere due metriche mIoU, mAP, F1 score? e 